Imports

In [0]:
from pyspark.sql import functions as F

In [0]:
LABOR_SILVER_TABLE = "databricks_project1.silver.labor_position"
LABOR_GOLD_TABLE = "databricks_project1.gold.dim_labor_position"

Read silver

In [0]:
labor_silver_df = spark.table(
    "databricks_project1.silver.labor_position"
)

display(labor_silver_df)

In [0]:
print(f"Silver row count: {labor_silver_df.count()}")

create gold dataframes`

In [0]:
final_labor_df = (
    labor_silver_df
    .select(
        F.col("Labor_Position_Code").cast("int").alias("Labor_Position_Code"),
        F.trim(F.col("Labor_Position_Desc")).alias("Labor_Position_Desc"),
        F.col("ingestion_timestamp"),
        F.col("source_file")
    )
    .dropDuplicates(["Labor_Position_Code"])
)

print(f"Final Dim_LaborPosition count: {final_labor_df.count()}")

display(final_labor_df)

write to gold

In [0]:
(
    final_labor_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "databricks_project1.gold.dim_labor_position"
    )
)

verify gold tables

In [0]:
gold_labor_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

print(f"Gold row count: {gold_labor_df.count()}")

display(gold_labor_df)

verify the null audit columsn

In [0]:
null_audit_count = gold_labor_df.filter(
    F.col("ingestion_timestamp").isNull() |
    F.col("source_file").isNull()
).count()

print(f"Rows with NULL audit columns: {null_audit_count}")

verify duplicate labor codes

In [0]:
duplicate_count = (
    gold_labor_df
    .groupBy("Labor_Position_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate Labor_Position_Code count: {duplicate_count}")

In [0]:
final_labor_df = spark.table(LABOR_GOLD_TABLE)

print("Final Dim_LaborPosition count:", final_labor_df.count())

final_labor_df.printSchema()

display(final_labor_df)